# Transient Oseen Flow Around a Cylinder — Documentation

This notebook documents the `oseen_trans_cyl.jl` script: mathematical background, finite element formulation, boundary conditions, assembly routines, and time integration. Each section explains the relevant equations, then shows the corresponding Julia code. A full runnable script is provided at the end.

## 1. Governing equations and Oseen linearization

We start from the incompressible Navier-Stokes equations in strong form:

$$
\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u} \cdot \nabla)\mathbf{u} - \nu \nabla^2 \mathbf{u} + \nabla p = \mathbf{0},
$$

$$
\nabla \cdot \mathbf{u} = 0.
$$

The Oseen approximation linearizes the convective term around a fixed reference velocity $\mathbf{w}$, replacing $(\mathbf{u} \cdot \nabla)\mathbf{u}$ with $(\mathbf{w} \cdot \nabla)\mathbf{u}$.

The corresponding weak form is:

Find $\mathbf{u}, p$ such that for all test functions $\mathbf{v}, q$,

$$
\int_\Omega \frac{\partial \mathbf{u}}{\partial t} \cdot \mathbf{v}\, d\Omega
+ \nu \int_\Omega \nabla \mathbf{u} : \nabla \mathbf{v}\, d\Omega
+ \int_\Omega (\mathbf{w} \cdot \nabla \mathbf{u}) \cdot \mathbf{v}\, d\Omega
- \int_\Omega p \, \nabla \cdot \mathbf{v}\, d\Omega
+ \int_\Omega q \, \nabla \cdot \mathbf{u}\, d\Omega = 0.
$$

This yields the semi-discrete system $\mathbf{M} \frac{d\mathbf{U}}{dt} = \mathbf{K}\mathbf{U}$, where $\mathbf{M}$ is the mass matrix and $\mathbf{K}$ is the Oseen operator.

### Imports and physical parameters

The script uses Ferrite for finite elements, Gmsh for meshing, and OrdinaryDiffEq/Rodas5P for time integration. We define the kinematic viscosity $\nu$ and a reference velocity $\mathbf{w}$ for the Oseen convective term.

In [ ]:
using Ferrite, SparseArrays, BlockArrays, LinearAlgebra, WriteVTK
using DiffEqBase
using OrdinaryDiffEqRosenbrock: Rodas5P

# Kinematic viscosity
ν = 1.0 / 1000.0 # kinematic viscosity

# Oseen reference velocity (can be chosen as mean inlet velocity)
w_oseen = Vec((0.0, 0.0))

## 2. Mesh generation and geometry setup

The domain is a rectangular channel with a circular obstacle (cylinder). The script constructs the geometry using Gmsh, performs a boolean cut to subtract the circle from the rectangle, identifies the physical boundary groups, and generates a 2D mesh.

In [ ]:
using FerriteGmsh
using FerriteGmsh: Gmsh
Gmsh.initialize()
gmsh.option.set_number("General.Verbosity", 2)
dim = 2

# Build the channel geometry with a circular obstacle
rect_tag = gmsh.model.occ.add_rectangle(0, 0, 0, 1.1, 0.41)
circle_tag = gmsh.model.occ.add_circle(0.2, 0.2, 0, 0.05)
circle_curve_tag = gmsh.model.occ.add_curve_loop([circle_tag])
circle_surf_tag = gmsh.model.occ.add_plane_surface([circle_curve_tag])
gmsh.model.occ.cut([(dim, rect_tag)], [(dim, circle_surf_tag)])
gmsh.model.occ.synchronize()

# Identify boundary entity tags robustly (bounding boxes)
ε = 1e-3
get_tags(bb...) = [t for (_, t) in gmsh.model.getEntitiesInBoundingBox(bb..., 1)]

bottom_tags = get_tags(-ε, -ε, -ε,  1.1+ε,      ε,  ε)
top_tags    = get_tags(-ε,  0.41-ε, -ε,  1.1+ε,  0.41+ε,  ε)
left_tags   = get_tags(-ε, -ε, -ε,      ε,  0.41+ε,  ε)
right_tags  = get_tags(1.1-ε, -ε, -ε,  1.1+ε,  0.41+ε,  ε)
hole_tags   = get_tags(0.2-0.06, 0.2-0.06, -ε,  0.2+0.06,  0.2+0.06,  ε)

# Remove overlapping tags and create physical groups
hole_tags = setdiff(hole_tags, bottom_tags, top_tags, left_tags, right_tags)
gmsh.model.add_physical_group(dim - 1, bottom_tags, -1, "bottom")
gmsh.model.add_physical_group(dim - 1, left_tags,   -1, "left")
gmsh.model.add_physical_group(dim - 1, right_tags,  -1, "right")
gmsh.model.add_physical_group(dim - 1, top_tags,    -1, "top")
gmsh.model.add_physical_group(dim - 1, hole_tags,   -1, "hole")

gmsh.option.setNumber("Mesh.Algorithm", 11)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 20)
gmsh.option.setNumber("Mesh.MeshSizeMax", 0.05)

gmsh.model.mesh.generate(dim)
grid = togrid()
Gmsh.finalize()

## 3. Finite element spaces and DOF handler

We use Taylor-Hood $Q_2/Q_1$ quadrilateral elements: quadratic velocity ($Q_2$) and linear pressure ($Q_1$). The DOF handler defines the global numbering for the velocity and pressure fields.

In [ ]:
# Taylor-Hood Q2/Q1 elements
ip_v = Lagrange{RefQuadrilateral, 2}()^dim
qr   = QuadratureRule{RefQuadrilateral}(4)
cellvalues_v = CellValues(qr, ip_v)

ip_p = Lagrange{RefQuadrilateral, 1}()
cellvalues_p = CellValues(qr, ip_p)

# DOF handler and fields
dh = DofHandler(grid)
add!(dh, :v, ip_v)
add!(dh, :p, ip_p)
close!(dh)

## 4. Boundary conditions

The script enforces no-slip conditions on the walls and cylinder, a time-ramped parabolic inflow on the left boundary, and a free outlet on the right. The inlet profile is

$$v_{\text{in}}(t) = \min(t\,U_{\max}, U_{\max}),$$

$$u_{\text{in}}(y,t) = v_{\text{in}}(t)\,\frac{4y(H-y)}{H^2}.$$

Here $H$ is the channel height and $U_{\max}$ is the maximum inlet speed used for ramping.

In [ ]:
# Constraint handler for Dirichlet BCs
ch = ConstraintHandler(dh)

noslip_facet_names = ["top", "bottom", "hole"]
∂Ω_noslip = union(getfacetset.((grid,), noslip_facet_names)...)
noslip_bc = Dirichlet(:v, ∂Ω_noslip, (x, t) -> Vec((0.0, 0.0)), [1, 2])
add!(ch, noslip_bc)

# Parabolic inflow (ramp up to 1.5 m/s)
∂Ω_inflow = getfacetset(grid, "left")
vᵢₙ(t) = min(t * 1.5, 1.5)
parabolic_inflow_profile(x, t) = Vec((4 * vᵢₙ(t) * x[2] * (0.41 - x[2]) / 0.41^2, 0.0))
inflow_bc = Dirichlet(:v, ∂Ω_inflow, parabolic_inflow_profile, [1, 2])
add!(ch, inflow_bc)

# Close and initialize BCs
close!(ch)
update!(ch, 0.0)

## 5. Mass matrix assembly

The velocity mass matrix entries are

$$M_{ij} = \int_\Omega \boldsymbol{\phi}_i \cdot \boldsymbol{\phi}_j\, d\Omega,$$

where $\boldsymbol{\phi}_i$ are the velocity shape functions.

Only the velocity block is non-zero because pressure has no time derivative in the incompressible formulation.

The mass matrix has the block structure

$$\mathbf{M} = \begin{bmatrix} \mathbf{M}_{vv} & \mathbf{0} \\ \mathbf{0} & \mathbf{0} \end{bmatrix}.$$

In [ ]:
function assemble_mass_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, M::SparseMatrixCSC, dh::DofHandler)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs   = n_basefuncs_v + n_basefuncs_p
    v▄, p▄ = 1, 2
    Mₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs),
                      [n_basefuncs_v, n_basefuncs_p],
                      [n_basefuncs_v, n_basefuncs_p])

    mass_assembler = start_assemble(M)
    for cell in CellIterator(dh)
        fill!(Mₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)

        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)
            for i in 1:n_basefuncs_v
                φᵢ = shape_value(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    φⱼ = shape_value(cellvalues_v, q_point, j)
                    Mₑ[BlockIndex((v▄, v▄), (i, j))] += φᵢ ⋅ φⱼ * dΩ
                end
            end
        end
        assemble!(mass_assembler, celldofs(cell), Mₑ)
    end
    return M
end

## 6. Oseen stiffness matrix assembly

From the weak form, the Oseen operator leads to a block structure with velocity-velocity, velocity-pressure, and pressure-velocity coupling. The velocity-velocity block is

$$K^{vv}_{ij} = -\nu \int_\Omega \nabla \phi_i : \nabla \phi_j\, d\Omega - \int_\Omega \phi_i \cdot (\mathbf{w} \cdot \nabla \phi_j)\, d\Omega.$$

The pressure coupling blocks are

$$K^{vp}_{ij} = \int_\Omega (\nabla \cdot \phi_i)\, \psi_j\, d\Omega, \qquad K^{pv}_{ij} = \int_\Omega \psi_i\,(\nabla \cdot \phi_j)\, d\Omega.$$

The saddle-point matrix is

$$\mathbf{K} = \begin{bmatrix} -\nu \mathbf{A} & \mathbf{B}^T \\ \mathbf{B} & \mathbf{0} \end{bmatrix}.$$

In [ ]:
function assemble_oseen_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, ν, w, K::SparseMatrixCSC, dh::DofHandler)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs   = n_basefuncs_v + n_basefuncs_p
    v▄, p▄ = 1, 2
    Kₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs),
                      [n_basefuncs_v, n_basefuncs_p],
                      [n_basefuncs_v, n_basefuncs_p])

    stiffness_assembler = start_assemble(K)
    for cell in CellIterator(dh)
        fill!(Kₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        Ferrite.reinit!(cellvalues_p, cell)

        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)

            # Velocity-velocity block: viscous + Oseen convection
            for i in 1:n_basefuncs_v
                ∇φᵢ = shape_gradient(cellvalues_v, q_point, i)
                φᵢ = shape_value(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    ∇φⱼ = shape_gradient(cellvalues_v, q_point, j)
                    φⱼ = shape_value(cellvalues_v, q_point, j)
                    Kₑ[BlockIndex((v▄, v▄), (i, j))] -= (ν * (∇φᵢ ⊡ ∇φⱼ) + φᵢ ⋅ (∇φⱼ ⋅ w)) * dΩ
                end
            end

            # Pressure-velocity coupling blocks
            for j in 1:n_basefuncs_p
                ψ = shape_value(cellvalues_p, q_point, j)
                for i in 1:n_basefuncs_v
                    divφ = shape_divergence(cellvalues_v, q_point, i)
                    Kₑ[BlockIndex((v▄, p▄), (i, j))] += (divφ * ψ) * dΩ
                    Kₑ[BlockIndex((p▄, v▄), (j, i))] += (ψ * divφ) * dΩ
                end
            end
        end
        assemble!(stiffness_assembler, celldofs(cell), Kₑ)
    end
    return K
end

## 7. Time integration setup and RHS

We solve the differential-algebraic equation system

$$\mathbf{M} \frac{d\mathbf{U}}{dt} = \mathbf{K}\mathbf{U}.$$

For this linear Oseen problem, the right-hand side is simply $\mathbf{K}\mathbf{U}$ and the Jacobian is constant: $\mathbf{J} = \mathbf{K}$.

The script uses `Rodas5P`, a Rosenbrock-type method with a mass matrix, to integrate in time.

In [ ]:
# Time integration parameters
T      = 20.0
Δt₀    = 0.001
Δt_save = 0.1

# Allocate and assemble matrices
M = allocate_matrix(dh)
M = assemble_mass_matrix(cellvalues_v, cellvalues_p, M, dh)

K = allocate_matrix(dh)
K = assemble_oseen_matrix(cellvalues_v, cellvalues_p, ν, w_oseen, K, dh)

u₀ = zeros(ndofs(dh))
apply!(u₀, ch)

jac_sparsity = sparse(K)

# Apply BCs to mass matrix
apply!(M, ch)

# Parameter struct used by ODE functions
struct RHSparams
    K::SparseMatrixCSC
    ch::ConstraintHandler
    dh::DofHandler,
    u::Vector
end
p = RHSparams(K, ch, dh, cellvalues_v, copy(u₀))

### RHS and Jacobian functions

For the linear Oseen problem,

$$\mathbf{M}\dot{\mathbf{u}} = \mathbf{K}\mathbf{u}, \qquad f(\mathbf{u}, t) = \mathbf{K}\mathbf{u}, \qquad \mathbf{J} = \mathbf{K}.$$

The functions `oseen!` and `oseen_jac!` apply the boundary conditions and compute the residual and Jacobian, respectively.

In [ ]:
function ferrite_limiter!(u, _, p, t)
    update!(p.ch, t)
    return apply!(u, p.ch)
end

function oseen!(du, u_uc, p::RHSparams, t)
    (; K, ch, u) = p
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    mul!(du, K, u)
    return
end

function oseen_jac!(J, u_uc, p::RHSparams, t)
    (; K, ch, u) = p
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    nonzeros(J) .= nonzeros(K)
    return apply!(J, ch)
end

## 8. Problem build, integrator, and output

The ODE problem is created with the mass matrix and Jacobian prototype for sparse operations. The integrator writes VTK output for each saved time step for visualization in ParaView.

In [ ]:
# Norm that only uses free DOFs
struct FreeDofErrorNorm
    ch::ConstraintHandler
end
(fe_norm::FreeDofErrorNorm)(u::Union{AbstractFloat, Complex}, t) = DiffEqBase.ODE_DEFAULT_NORM(u, t)
(fe_norm::FreeDofErrorNorm)(u::AbstractArray, t) = DiffEqBase.ODE_DEFAULT_NORM(u[fe_norm.ch.free_dofs], t)

rhs = ODEFunction(oseen!, mass_matrix = M; jac = oseen_jac!, jac_prototype = jac_sparsity)
problem = ODEProblem(rhs, u₀, (0.0, T), p)

timestepper = Rodas5P(autodiff = false, step_limiter! = ferrite_limiter!)
integrator = init(
    problem, timestepper; initializealg = NoInit(), dt = Δt₀,
    adaptive = true, abstol = 1.0e-4, reltol = 1.0e-5,
    progress = true, progress_steps = 1,
    verbose = true, internalnorm = FreeDofErrorNorm(ch), d_discontinuities = [1.0]
)

# Time-stepping loop with VTK output
pvd = paraview_collection("oseen_trans_cyl_0")
for (step, (u, t)) in enumerate(intervals(integrator))
    VTKGridFile("oseen_trans_cyl_0-$step", dh) do vtk
        write_solution(vtk, dh, u)
        pvd[t] = vtk
    end
end
vtk_save(pvd)
println("Done!")

## 9. Full script (runnable)

Below is the entire `oseen_trans_cyl.jl` script in one cell so the notebook can run standalone if executed from top to bottom. Use this single cell to run the full simulation in a fresh kernel; it duplicates the fragments above.

In [ ]:
# Full `oseen_trans_cyl.jl` script — copy of original file
# Transient Oseen flow around a cylinder
# Based on the transient Navier-Stokes tutorial (test_cyl_ns.jl) with the convective 
# term replaced by the linearized Oseen approximation.
#
# Governing equation:
#   M * du/dt = K * u
# where K is the Oseen operator (viscous diffusion + Oseen convection + pressure-velocity coupling)
# and M is the velocity mass matrix.

using Ferrite, SparseArrays, BlockArrays, LinearAlgebra, WriteVTK

using DiffEqBase
using OrdinaryDiffEqRosenbrock: Rodas5P

ν = 1.0 / 1000.0 # kinematic viscosity

# Oseen reference velocity: mean of the parabolic inflow profile
#   u(y) = 4 * U_max * y * (H - y) / H²  →  ū = (2/3) * U_max = (2/3) * 1.5 = 1.0 m/s
# This is the characteristic velocity used to linearize the convective term.
w_oseen = Vec((0.0, 0.0))

using FerriteGmsh
using FerriteGmsh: Gmsh
Gmsh.initialize()
gmsh.option.set_number("General.Verbosity", 2)
dim = 2

# Build the channel geometry with a circular obstacle
rect_tag = gmsh.model.occ.add_rectangle(0, 0, 0, 1.1, 0.41)
circle_tag = gmsh.model.occ.add_circle(0.2, 0.2, 0, 0.05)
circle_curve_tag = gmsh.model.occ.add_curve_loop([circle_tag])
circle_surf_tag = gmsh.model.occ.add_plane_surface([circle_curve_tag])
gmsh.model.occ.cut([(dim, rect_tag)], [(dim, circle_surf_tag)])

gmsh.model.occ.synchronize()

# Use bounding-box queries to identify boundary curves robustly after the boolean cut
ε = 1e-3
get_tags(bb...) = [t for (_, t) in gmsh.model.getEntitiesInBoundingBox(bb..., 1)]

bottom_tags = get_tags(-ε, -ε, -ε,  1.1+ε,      ε,  ε)
top_tags    = get_tags(-ε,  0.41-ε, -ε,  1.1+ε,  0.41+ε,  ε)
left_tags   = get_tags(-ε, -ε, -ε,      ε,  0.41+ε,  ε)
right_tags  = get_tags(1.1-ε, -ε, -ε,  1.1+ε,  0.41+ε,  ε)
hole_tags   = get_tags(0.2-0.06, 0.2-0.06, -ε,  0.2+0.06,  0.2+0.06,  ε)

hole_tags = setdiff(hole_tags, bottom_tags, top_tags, left_tags, right_tags)

gmsh.model.add_physical_group(dim - 1, bottom_tags, -1, "bottom")
gmsh.model.add_physical_group(dim - 1, left_tags,   -1, "left")
gmsh.model.add_physical_group(dim - 1, right_tags,  -1, "right")
gmsh.model.add_physical_group(dim - 1, top_tags,    -1, "top")
gmsh.model.add_physical_group(dim - 1, hole_tags,   -1, "hole")

gmsh.option.setNumber("Mesh.Algorithm", 11)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 20)
gmsh.option.setNumber("Mesh.MeshSizeMax", 0.05)

gmsh.model.mesh.generate(dim)
grid = togrid()
Gmsh.finalize()

# Taylor-Hood Q2/Q1 elements
ip_v = Lagrange{RefQuadrilateral, 2}()^dim
qr   = QuadratureRule{RefQuadrilateral}(4)
cellvalues_v = CellValues(qr, ip_v)

ip_p = Lagrange{RefQuadrilateral, 1}()
cellvalues_p = CellValues(qr, ip_p)

dh = DofHandler(grid)
add!(dh, :v, ip_v)
add!(dh, :p, ip_p)
close!(dh)

# Boundary conditions
ch = ConstraintHandler(dh)

noslip_facet_names = ["top", "bottom", "hole"]
∂Ω_noslip = union(getfacetset.((grid,), noslip_facet_names)...)
noslip_bc = Dirichlet(:v, ∂Ω_noslip, (x, t) -> Vec((0.0, 0.0)), [1, 2])
add!(ch, noslip_bc)

∂Ω_inflow = getfacetset(grid, "left")
vᵢₙ(t) = min(t * 1.5, 1.5)
parabolic_inflow_profile(x, t) = Vec((4 * vᵢₙ(t) * x[2] * (0.41 - x[2]) / 0.41^2, 0.0))
inflow_bc = Dirichlet(:v, ∂Ω_inflow, parabolic_inflow_profile, [1, 2])
add!(ch, inflow_bc)

∂Ω_free = getfacetset(grid, "right")

close!(ch)
update!(ch, 0.0)

# Mass matrix assembly
function assemble_mass_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, M::SparseMatrixCSC, dh::DofHandler)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs   = n_basefuncs_v + n_basefuncs_p
    v▄, p▄ = 1, 2
    Mₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs),
                      [n_basefuncs_v, n_basefuncs_p],
                      [n_basefuncs_v, n_basefuncs_p])

    mass_assembler = start_assemble(M)
    for cell in CellIterator(dh)
        fill!(Mₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)

        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)
            for i in 1:n_basefuncs_v
                φᵢ = shape_value(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    φⱼ = shape_value(cellvalues_v, q_point, j)
                    Mₑ[BlockIndex((v▄, v▄), (i, j))] += φᵢ ⋅ φⱼ * dΩ
                end
            end
        end
        assemble!(mass_assembler, celldofs(cell), Mₑ)
    end
    return M
end

# Oseen stiffness assembly
function assemble_oseen_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, ν, w, K::SparseMatrixCSC, dh::DofHandler)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs   = n_basefuncs_v + n_basefuncs_p
    v▄, p▄ = 1, 2
    Kₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs),
                      [n_basefuncs_v, n_basefuncs_p],
                      [n_basefuncs_v, n_basefuncs_p])

    stiffness_assembler = start_assemble(K)
    for cell in CellIterator(dh)
        fill!(Kₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        Ferrite.reinit!(cellvalues_p, cell)

        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)

            for i in 1:n_basefuncs_v
                ∇φᵢ = shape_gradient(cellvalues_v, q_point, i)
                φᵢ = shape_value(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    ∇φⱼ = shape_gradient(cellvalues_v, q_point, j)
                    φⱼ = shape_value(cellvalues_v, q_point, j)
                    Kₑ[BlockIndex((v▄, v▄), (i, j))] -= (ν * (∇φᵢ ⊡ ∇φⱼ) + φᵢ ⋅ (∇φⱼ ⋅ w)) * dΩ
                end
            end

            for j in 1:n_basefuncs_p
                ψ = shape_value(cellvalues_p, q_point, j)
                for i in 1:n_basefuncs_v
                    divφ = shape_divergence(cellvalues_v, q_point, i)
                    Kₑ[BlockIndex((v▄, p▄), (i, j))] += (divφ * ψ) * dΩ
                    Kₑ[BlockIndex((p▄, v▄), (j, i))] += (ψ * divφ) * dΩ
                end
            end
        end
        assemble!(stiffness_assembler, celldofs(cell), Kₑ)
    end
    return K
end

# Time integration parameters
T      = 20.0
Δt₀    = 0.001
Δt_save = 0.1

# Allocate and assemble the system matrices
M = allocate_matrix(dh)
M = assemble_mass_matrix(cellvalues_v, cellvalues_p, M, dh)

K = allocate_matrix(dh)
K = assemble_oseen_matrix(cellvalues_v, cellvalues_p, ν, w_oseen, K, dh)

u₀ = zeros(ndofs(dh))
apply!(u₀, ch)

jac_sparsity = sparse(K)

# Apply BCs to mass matrix
apply!(M, ch)

# ODE parameter struct
struct RHSparams
    K::SparseMatrixCSC
    ch::ConstraintHandler
    dh::DofHandler
    cellvalues_v::CellValues
    u::Vector
end

p = RHSparams(K, ch, dh, cellvalues_v, copy(u₀))

function ferrite_limiter!(u, _, p, t)
    update!(p.ch, t)
    return apply!(u, p.ch)
end

function oseen!(du, u_uc, p::RHSparams, t)
    (; K, ch, u) = p

    u .= u_uc
    update!(ch, t)
    apply!(u, ch)

    mul!(du, K, u)
    return
end

function oseen_jac!(J, u_uc, p::RHSparams, t)
    (; K, ch, u) = p

    u .= u_uc
    update!(ch, t)
    apply!(u, ch)

    nonzeros(J) .= nonzeros(K)

    return apply!(J, ch)
end

# Build and solve the ODE problem
struct FreeDofErrorNorm
    ch::ConstraintHandler
end
(fe_norm::FreeDofErrorNorm)(u::Union{AbstractFloat, Complex}, t) = DiffEqBase.ODE_DEFAULT_NORM(u, t)
(fe_norm::FreeDofErrorNorm)(u::AbstractArray, t) = DiffEqBase.ODE_DEFAULT_NORM(u[fe_norm.ch.free_dofs], t)

rhs = ODEFunction(oseen!, mass_matrix = M; jac = oseen_jac!, jac_prototype = jac_sparsity)
problem = ODEProblem(rhs, u₀, (0.0, T), p)

timestepper = Rodas5P(autodiff = false, step_limiter! = ferrite_limiter!)

integrator = init(
    problem, timestepper; initializealg = NoInit(), dt = Δt₀,
    adaptive = true, abstol = 1.0e-4, reltol = 1.0e-5,
    progress = true, progress_steps = 1,
    verbose = true, internalnorm = FreeDofErrorNorm(ch), d_discontinuities = [1.0]
)

# Time-stepping loop with VTK output
pvd = paraview_collection("oseen_trans_cyl_0")
for (step, (u, t)) in enumerate(intervals(integrator))
    VTKGridFile("oseen_trans_cyl_0-$step", dh) do vtk
        write_solution(vtk, dh, u)
        pvd[t] = vtk
    end
end
vtk_save(pvd)

println("Done!")

---

Notes:
- You can run the notebook top-to-bottom in a Julia kernel (IJulia).
- If you want to run the full script cell, ensure the required packages (`Ferrite`, `FerriteGmsh`, `WriteVTK`, `OrdinaryDiffEqRosenbrock`) are installed and Gmsh is available on your PATH.
- The full script duplicates the earlier fragments, so you may run either the fragments stepwise or the single full script cell.